In [1]:
# Importación de librerías
import pandas as pd
import mysql.connector
from mysql.connector import Error

## PARTE A – Análisis y limpieza del archivo vgsales.csv

### 1. Carga el contenido del archivo.

In [2]:
df = pd.read_csv('vgsales.csv')
df

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37
...,...,...,...,...,...,...,...,...,...,...,...
16593,16596,Woody Woodpecker in Crazy Castle 5,GBA,2002.0,Platform,Kemco,0.01,0.00,0.00,0.00,0.01
16594,16597,Men in Black II: Alien Escape,GC,2003.0,Shooter,Infogrames,0.01,0.00,0.00,0.00,0.01
16595,16598,SCORE International Baja 1000: The Official Game,PS2,2008.0,Racing,Activision,0.00,0.00,0.00,0.00,0.01
16596,16599,Know How 2,DS,2010.0,Puzzle,7G//AMES,0.00,0.01,0.00,0.00,0.01


### 2. ¿Cuántos valores nulos tiene cada columna?

In [3]:
df.isnull().sum()

Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64

### 3. Crea una copia del DataFrame e imputa los valores faltantes:
#### o Para columnas numéricas, usa la mediana.
#### o Para columnas categóricas, usa la moda.

In [4]:
# Copia del dataframe
df_copy = df.copy()
df_copy.info() # Para ver cuales son las numericas

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entries, 0 to 16597
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Rank          16598 non-null  int64  
 1   Name          16598 non-null  object 
 2   Platform      16598 non-null  object 
 3   Year          16327 non-null  float64
 4   Genre         16598 non-null  object 
 5   Publisher     16540 non-null  object 
 6   NA_Sales      16598 non-null  float64
 7   EU_Sales      16598 non-null  float64
 8   JP_Sales      16598 non-null  float64
 9   Other_Sales   16598 non-null  float64
 10  Global_Sales  16598 non-null  float64
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


In [5]:
# Imputar valores faltantes
moda_year = df_copy['Year'].mode()[0] # Year es categorica
df_copy['Year'] = df_copy['Year'].fillna(moda_year)

moda_publisher = df_copy['Publisher'].mode()[0] # Publisher es categorica
df_copy['Publisher'] = df_copy['Publisher'].fillna(moda_publisher)
df_copy.isnull().sum()

Rank            0
Name            0
Platform        0
Year            0
Genre           0
Publisher       0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64

### 4. Elimina cualquier registro que tenga:
#### o Año (Year) mayor al año actual.
#### o Ventas (Global_Sales) menores o iguales a cero.

In [6]:
condicion_edad = (df_copy['Year'] <= 2026)
condicion_ventas = (df_copy['Global_Sales'] > 0)
# Filtrado de registros
df_copy = df_copy[condicion_edad & condicion_ventas]

### 5. Valida que las siguientes columnas contengan valores válidos:
#### o Genre: Solo deben existir géneros comunes como: "Action", "Adventure", "RPG", "Shooter", "Sports", "Platform", "Puzzle".
#### o Platform: No debe haber valores vacíos.

In [7]:
# Cuenta de los que no estan 
df_copy = df_copy[df_copy['Genre'].isin(["Action", "Adventure", "RPG", "Shooter", "Sports", "Platform", "Puzzle"])]

In [8]:
# Validación de Plataform
# Otra forma era con df_copy = df_copy[df_copy['Platform'].notnull()]
df_copy = df_copy[~df_copy['Platform'].isnull()]

### 6. Crea una nueva columna llamada Decade que clasifique cada juego según su año de lanzamiento en:
#### o "1980s", "1990s", "2000s", "2010s", "2020s"

In [9]:
limites = [1980, 1990, 2000, 2010, 2020, 2030]
etiquetas = ["1980s", "1990s", "2000s", "2010s", "2020s"]
df_copy['Decade'] = pd.cut(df_copy['Year'], bins=limites, labels=etiquetas)
df_copy[['Name', 'Year','Decade']]

,Name,Year,Decade
0,Wii Sports,2006.0,2000s
1,Super Mario Bros.,1985.0,1980s
3,Wii Sports Resort,2009.0,2000s
5,Tetris,1989.0,1980s
6,New Super Mario Bros.,2006.0,2000s
...,...,...,...
16591,Myst IV: Revelation,2004.0,2000s
16593,Woody Woodpecker in Crazy Castle 5,2002.0,2000s
16594,Men in Black II: Alien Escape,2003.0,2000s
16596,Know How 2,2010.0,2000s


### 7. Diseña un modelo de base de datos relacional para registrar:
#### o GAME: Id, IdPlatform, Name, Year, Genre, Publisher, Global_Sales.
#### o PLATFORM: Id, Name
### 8. Prepara los datos para cargarlos en la base de datos.

In [10]:
# Realizar la conexion
config_db = {
    'host':'localhost',
    'user': 'root',
    'password': 'password', # 'password': 'root' 
    'port': '3307', #'port': '3306' 
    'database': 'recu_unidad2',
    'charset': 'utf8mb4',
    'use_unicode': True
}

try:
    connection = mysql.connector.connect(**config_db)
    if connection.is_connected():
        print('Conexion exitosa')
except Error as e:
   print(f'Error al conectar: {e}') 

Conexion exitosa


In [11]:
# Insertar los datos de plataformas
plataformas = df_copy.groupby('Platform').size().reset_index(name='cantidad')

### 9. Carga los datos a una base de datos relacional en MySQL.

In [12]:
cursor = connection.cursor()
for i, plataforma in plataformas.iterrows():
    cursor.execute('INSERT IGNORE INTO PLATFORM(Name) VALUES(%s)', (plataforma['Platform'],))
connection.commit()

In [13]:
# Insertar datos de juegos
for i, juego in df_copy.iterrows():
    cursor.execute('SELECT Id FROM PLATFORM WHERE Name = %s', (juego['Platform'],))
    resultado = cursor.fetchone()
    cursor.fetchall()

    cursor.execute('INSERT IGNORE INTO GAME(IdPlatform, Name, Year, Genre, Publisher, Global_Sales) VALUES(%s, %s, %s, %s, %s, %s)', 
                   (resultado[0], juego['Name'], juego['Year'], juego['Genre'], juego['Publisher'], juego['Global_Sales']))
    cursor.fetchall()
connection.commit()

## PARTE B – Consultas analíticas

### 1. ¿Cuáles son las plataformas que se encuentran en el archivo?

In [14]:
df_copy.groupby('Platform').size().reset_index(name='cantidad')['Platform'].tolist()

['2600',
 '3DO',
 '3DS',
 'DC',
 'DS',
 'GB',
 'GBA',
 'GC',
 'GEN',
 'GG',
 'N64',
 'NES',
 'NG',
 'PC',
 'PS',
 'PS2',
 'PS3',
 'PS4',
 'PSP',
 'PSV',
 'SAT',
 'SCD',
 'SNES',
 'TG16',
 'Wii',
 'WiiU',
 'X360',
 'XB',
 'XOne']

### 2. ¿Cuántas plataformas se encuentran en el archivo?

In [15]:
# Forma 1
df_copy.groupby('Platform').size().reset_index(name='cantidad')['Platform'].count()

np.int64(29)

In [16]:
# Forma 2
df_copy.groupby('Platform').size().reset_index(name='cantidad').shape[0]

29

### 3. ¿Cuál es la empresa que tiene más registros en el archivo?

In [17]:
# Se agrupan las empresas
empresas_cantidad = df_copy.groupby('Publisher').size().reset_index(name='Cantidad')
# Se encuentra la cantidad mayor de registros
cantidad_mayor = empresas_cantidad['Cantidad'].max()
# Se imprime el que tiene la cantidad mayor
empresas_cantidad[empresas_cantidad['Cantidad'] == cantidad_mayor]['Publisher']

116    Electronic Arts
Name: Publisher, dtype: object

### 4. ¿Cuál fue el promedio de ventas globales de Nintendo?

In [18]:
df_copy[df_copy['Publisher'] == 'Nintendo']['Global_Sales'].mean()

np.float64(2.6344356955380577)